# Mecánica, datos y reproducibilidad

Reúne geometría de microtúbulos, resortes, fuerzas, movimiento browniano, serialización y prácticas reproducibles para los experimentos.

> **Nota científica:** KineSIM es un prototipo de modelado cinético fenomenológico. Los parámetros iniciales son ilustrativos y los mapas de energía no están calibrados experimentalmente para kinesina.

## Resultados de aprendizaje

Este notebook reúne herramientas auxiliares para ampliar KineSIM: una pista molecular discretizada, modelos sencillos de elasticidad y arrastre, incrementos brownianos y almacenamiento de trayectorias. También introduce prácticas para documentar las condiciones de un experimento computacional.

**Importante:** estos componentes mecánicos se estudian por separado. No están acoplados al algoritmo de Gillespie ni modifican actualmente la marcha de la kinesina.

## Instalación desde PyPI

Este notebook utiliza la distribución publicada de **KineSIM**. No modifica `sys.path` ni importa código desde la carpeta `src/`.

- Nombre de distribución: `kinesim` (PyPI no distingue mayúsculas).
- Namespace de importación: `protein_motor_sim` (se conserva por compatibilidad).
- Versión fijada para reproducibilidad: `0.1.2`.

Ejecuta la siguiente celda con acceso a Internet. Si la versión todavía no aparece en PyPI, primero debe publicarse con Twine. Para probar una subida en TestPyPI, consulta las instrucciones de instalación de este README; no mezcles los índices de TestPyPI y PyPI en una misma resolución de dependencias.


In [ ]:
%pip install --quiet "kinesim[notebooks]==0.1.2"


## 1. Objetivo

Integramos herramientas mecánicas auxiliares, geometría de la pista molecular y almacenamiento de resultados. Estos componentes son útiles para experimentos computacionales, aunque los modelos mecánicos todavía no están acoplados al integrador cinético.

### Mapa del laboratorio

1. **Geometría:** convertir índices de sitios en coordenadas longitudinales.
2. **Elasticidad:** calcular la fuerza restauradora de un resorte lineal.
3. **Arrastre:** calcular la resistencia viscosa lineal de una carga puntual.
4. **Difusión:** generar incrementos aleatorios con la varianza browniana esperada.
5. **Datos:** guardar y recuperar una trayectoria en JSON y exportar sus registros a CSV.

En todas las celdas se especifican unidades para distinguir posiciones en nm de posiciones mecánicas en m.

In [ ]:
from importlib.metadata import version as distribution_version
print("KineSIM distribution:", distribution_version("kinesim"))

import csv
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from protein_motor_sim import Environment, GillespieSimulator, Kinesin1
from protein_motor_sim.core.track import Track
from protein_motor_sim.mechanics.elasticity import HookeanSpring
from protein_motor_sim.mechanics.forces import Force, net_force
from protein_motor_sim.mechanics.cargo import Cargo
from protein_motor_sim.mechanics.brownian import brownian_displacement
from protein_motor_sim.io.serialization import save_trajectory, load_trajectory

## 2. Geometría de un microtúbulo simplificado

`Track` describe una pista unidimensional con sitios periódicos. El espaciamiento de 8.2 nm es una escala nominal de repetición longitudinal usada aquí para la representación simplificada.

### Geometría de una pista unidimensional

La clase `Track` representa una sucesión periódica de sitios:

$$
x_n=x_0+pnd,
$$

con $x_0$ el origen en nm, $p\in\{-1,+1\}$ la polaridad y $d$ la separación entre sitios en nm. El valor nominal $d=8.2$ nm se usa como escala longitudinal de paso para este modelo simplificado.

La pista no reproduce la estructura cilíndrica del microtúbulo, sus 13 protofilamentos, la disposición helicoidal de tubulina ni todos los sitios de unión. Es una representación geométrica 1D.

In [ ]:
track = Track(name="microtubule", lattice_spacing_nm=8.2, polarity=1, origin_nm=0.0)
indices = np.arange(-2, 8)
positions = [track.site_position_nm(int(i)) for i in indices]
pd = list(zip(indices, positions))
print("(índice, posición nm):")
print(pd)
print("Índice más cercano a 25 nm:", track.nearest_site_index(25.0))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2))
ax.scatter(positions, np.zeros_like(positions), s=100)
for index, position in zip(indices, positions):
    ax.annotate(str(index), (position, 0), xytext=(0, 10), textcoords="offset points", ha="center")
ax.set_xlabel("Posición sobre la pista (nm)")
ax.set_yticks([])
ax.set_title("Sitios discretos del microtúbulo")
ax.grid(axis="x", alpha=0.2)
plt.show()

### Interpretar la geometría

Los puntos de la gráfica son sitios discretos y las etiquetas indican su índice entero. Para `polarity=+1`, los índices crecientes aumentan $x$; con `polarity=-1`, la orientación se invierte.

`nearest_site_index` proyecta una posición continua sobre el índice entero más próximo. Es una operación geométrica de redondeo, no un modelo de unión molecular ni una regla de ocupación estérica.

## 3. Elasticidad y fuerza de arrastre

La ley de Hooke aproxima un elemento elástico lineal. La carga usa un arrastre viscoso lineal. Son modelos auxiliares y no describen por sí solos la elasticidad estructural real del neck linker.

### Elasticidad y arrastre

### Resorte lineal

La ley de Hooke se escribe:

$$
F_{\mathrm{spring}}=-k(x-x_0),
\qquad U_{\mathrm{spring}}=\frac{1}{2}k(x-x_0)^2.
$$

La constante $k$ tiene unidades N/m, las longitudes se expresan en m y la fuerza resultante en N. El signo negativo indica que la fuerza restauradora se opone a la deformación.

### Arrastre lineal

Para una carga que se mueve en un régimen de arrastre lineal:

$$
F_{\mathrm{drag}}=-\gamma v,
$$

con $\gamma$ en N s/m y $v$ en m/s. La fuerza se opone a la velocidad. Esta ley es una aproximación de baja velocidad y no describe por sí sola la hidrodinámica compleja del citoplasma ni el acoplamiento real de una vesícula a kinesina.

In [ ]:
spring = HookeanSpring(stiffness_N_m=0.2e-3, rest_length_m=5e-9)
length_m = 6e-9
print("Fuerza restauradora (pN):", spring.force_N(length_m) * 1e12)

cargo = Cargo(position_m=0.0, drag_coefficient_N_s_m=2e-6)
velocity_m_s = 1e-6
print("Fuerza de arrastre (pN):", cargo.drag_force_N(velocity_m_s) * 1e12)

resultant = net_force(Force(2e-12, "motor"), Force(-0.5e-12, "load"))
print("Fuerza resultante (pN):", resultant.value_N * 1e12)

### Suma de fuerzas

`net_force` suma fuerzas escalares colineales con signo. El ejemplo supone que todas actúan sobre un mismo eje. Si las fuerzas no son colineales, se necesita una suma vectorial y no esta representación escalar.

## 4. Desplazamientos brownianos

Para una coordenada difusiva libre, el incremento tiene distribución normal con media cero y varianza `2 D dt`. La función genera un único incremento; no integra todavía una ecuación de Langevin con potencial y fuerzas.

### Desplazamientos brownianos

Para difusión libre en una dimensión, el incremento durante un intervalo $\Delta t$ se modela como:

$$
\Delta x=\sqrt{2D\Delta t}\,\xi,
\qquad \xi\sim\mathcal{N}(0,1).
$$

Por tanto:

$$
\mathbb{E}[\Delta x]=0,
\qquad \operatorname{Var}(\Delta x)=2D\Delta t.
$$

Aquí $D$ tiene unidades m$^2$/s, $\Delta t$ se expresa en s y el incremento queda en m. La función `brownian_displacement` genera un incremento independiente. No integra una ecuación de Langevin con fuerzas, potencial, condiciones de frontera o memoria hidrodinámica.

El valor de $D$ seleccionado en el ejemplo es ilustrativo y no corresponde a una medición de kinesina ni de una vesícula concreta.

In [ ]:
D = 1.0e-12  # m^2/s, valor ilustrativo
dt = 1.0e-3   # s
rng = np.random.default_rng(123)
steps_m = np.array([brownian_displacement(D, dt, rng) for _ in range(5000)])
print("Media muestral (m):", steps_m.mean())
print("Varianza muestral (m^2):", steps_m.var())
print("Varianza teórica (m^2):", 2 * D * dt)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(steps_m * 1e9, bins=45, density=True, alpha=0.7)
ax.set_xlabel("Incremento browniano (nm)")
ax.set_ylabel("Densidad")
ax.set_title("Distribución de desplazamientos brownianos")
plt.show()

### Qué comprobar en el histograma

El histograma se normaliza como densidad. Con un número suficientemente grande de incrementos debe aproximarse a una gaussiana centrada en cero. La media muestral fluctúa alrededor de cero y la varianza muestral fluctúa alrededor de $2D\Delta t$.

Aumentar el número de muestras mejora la estabilidad estadística del histograma, pero no cambia la distribución teórica elegida.

## 5. Guardar y recuperar una trayectoria

La serialización permite conservar los registros y analizarlos posteriormente. Se utiliza una carpeta `results/` en la raíz del proyecto.

In [ ]:
from pathlib import Path
OUTPUT_DIR = Path.cwd() / "kinesim_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

motor = Kinesin1()
environment = Environment(atp_concentration=1e-3, temperature_K=310.0)
trajectory = GillespieSimulator().run(motor, environment, t_max=0.4, seed=2026)
results_dir = OUTPUT_DIR
results_dir.mkdir(parents=True, exist_ok=True)
json_path = results_dir / "trajectory_example.json"
save_trajectory(trajectory, json_path)
reloaded = load_trajectory(json_path)
print("Archivo:", json_path)
print("Posiciones idénticas:", trajectory.positions_nm == reloaded.positions_nm)
print("Eventos idénticos:", trajectory.events == reloaded.events)
print("ATP consumido:", reloaded.atp_consumed_total)

### Serialización JSON

`save_trajectory` convierte los registros de `Trajectory` en un diccionario y los escribe en JSON. `load_trajectory` lee ese archivo y reconstruye los tiempos, posiciones, estados, eventos y el contador acumulado de ATP.

El archivo permite conservar la trayectoria para procesarla sin volver a ejecutar Gillespie. Sin embargo, la estructura JSON actual **no almacena automáticamente** toda la configuración experimental, la semilla, la versión de KineSIM ni los parámetros del motor. Para una trazabilidad completa, esos metadatos deben guardarse junto al archivo o añadirse en una futura ampliación del formato.

## 6. Guardar una tabla CSV

Además del JSON, podemos guardar un archivo tabular con una fila por registro temporal.

In [ ]:
csv_path = results_dir / "trajectory_example.csv"
with csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["time_s", "position_nm", "state", "event"])
    writer.writerows(zip(
        reloaded.times_s, reloaded.positions_nm, reloaded.states, reloaded.events
    ))
print("CSV guardado en:", csv_path)

### CSV y análisis tabular

El CSV guarda una fila por registro temporal y columnas para tiempo, posición, estado y evento. Es cómodo para abrir en hojas de cálculo o leer con Pandas. A diferencia del JSON, esta tabla no pretende reconstruir por sí sola todos los atributos de `Trajectory`; es una exportación tabular de los registros seleccionados.

## 7. Flujo reproducible de trabajo

Para que un experimento computacional sea reproducible, registra como mínimo: versión del código, parámetros del motor, entorno, tiempo máximo, número de realizaciones y semillas. Una semilla permite reproducir una trayectoria, pero la validación científica requiere además análisis estadístico y comparación con datos experimentales.

### Lista de comprobación para reproducibilidad

Para que otra persona pueda repetir un experimento, registra como mínimo:

- versión de KineSIM y versión de Python;
- parámetros del motor y del entorno (ATP, temperatura y fuerza);
- tiempo máximo y número de realizaciones;
- semilla maestra y, si corresponde, las semillas de cada trayectoria;
- definición exacta de los observables y tratamiento de trayectorias censuradas;
- dependencias y método de instalación.

Una semilla reproduce una realización pseudoaleatoria bajo las mismas condiciones de software; no demuestra que el modelo sea físicamente válido. La validación científica requiere contrastar observables con datos experimentales y documentar los supuestos.

## Referencia para difusión browniana

- Einstein, A. (1905). *Über die von der molekularkinetischen Theorie der Wärme geforderte Bewegung von in ruhenden Flüssigkeiten suspendierten Teilchen*. Annalen der Physik, 322(8), 549–560. [https://doi.org/10.1002/andp.19053220806](https://doi.org/10.1002/andp.19053220806)